Tutorial 6: Borders
===================

In the previous tutorials, the source-plane pixel grid perfectly mapped over the traced image-pixel $(y,x)$ coordinates
in the source plane. If these pixels mapped to a larger area in the source plane, its pixel-grid would automatically
increase its size so as to cover every source-plane coordinate.

In this tutorial, we will consider how the size of the pixelization grid is chosen and introduce the concept of a
border. The previous tutorial applied border relocation as a single step of the likelihood algebra -- here we see
why that step is needed.

__Contents__

- **Initial Setup:** we'll use the same strong lensing data as the previous tutorial, where.
- **Borders:** So, what is a border?
- **Wrap Up:** Summary of the script and next steps.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("howtolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autolens as al
import autolens.plot as aplt

__Initial Setup__

we'll use the same strong lensing data as the previous tutorial, where:

 - The lens galaxy's light is omitted.
 - The lens system uses an `Isothermal` galaxy mass profile and a separate `ExternalShear` field.
 - The source galaxy's light is an `Sersic`.

In [ ]:
dataset_name = "simple__no_lens_light"
dataset_path = Path("dataset") / "imaging" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/simulator/no_lens_light.py"],
        check=True,
    )

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    psf_path=dataset_path / "psf.fits",
    pixel_scales=0.1,
)

aplt.subplot_imaging_dataset(dataset=dataset)

dataset_unmasked = dataset

__Borders__

So, what is a border? In the image-plane, a border is the set of exterior pixels in a mask that are at its border.

Lets apply a circular mask to the image and plot the dataset -- the border is the ring of pixels at the mask's edge.

In [ ]:
mask_circular = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=2.5,
)
dataset = dataset.apply_mask(mask=mask_circular)

border = mask_circular.derive_grid.border


aplt.subplot_imaging_dataset(dataset=dataset)

As you can see, for a circular mask the border *is* the edge of the mask (the ring of black dots we're used to
seeing whenever we plot a mask).

For an annular mask, pixels on its inner edge are not a part of the border, whereas those on its outer edge are.

In [ ]:

mask_annular = al.Mask2D.circular_annular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    inner_radius=0.8,
    outer_radius=2.5,
)

dataset = dataset.apply_mask(mask=mask_annular)

border = mask_annular.derive_grid.border


aplt.subplot_imaging_dataset(dataset=dataset)

A border is therefore *only* the pixels at the exterior edge of a mask

So, what does a border actually do? To show you, we'll need to fit this image with a lens model and inversion. We'll 
use the same convenience function we used in the previous tutorial (to perform a quick source galaxy fit) with the 
options to input a mask and use a border.

In [ ]:


def perform_fit_with_source_galaxy_mask_and_border(
    dataset, source_galaxy, mask, settings
):
    dataset = dataset.apply_mask(mask=mask)

    lens_galaxy = al.Galaxy(
        redshift=0.5,
        mass=al.mp.Isothermal(
            centre=(0.0, 0.0),
            einstein_radius=1.6,
            ell_comps=al.convert.ell_comps_from(axis_ratio=0.9, angle=45.0),
        ),
    )
    field = al.MassField(
        redshift=0.5, shear=al.mp.ExternalShear(gamma_1=0.05, gamma_2=0.05)
    )

    tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy], fields=[field])

    return al.FitImaging(dataset=dataset, tracer=tracer, settings=settings)


Okay, so lets first look at the mapper without using a border, using the annular mask.

Note how we control the border via the `al.Settings` object's `use_border_relocator` input.

In [ ]:
# `PYAUTO_SMALL_DATASETS=1` (the CI smoke harness) caps the data to a 16 x 16 image, where a
# 40 x 40 mesh reconstructs 1600 source pixels from a few dozen image pixels. The mesh is
# capped alongside the data; a normal full-resolution run keeps the 40 x 40 grid described above.
mesh_shape = (
    al.util.dataset.SMALL_DATASETS_SHAPE_NATIVE
    if dataset.shape_native == al.util.dataset.SMALL_DATASETS_SHAPE_NATIVE
    else (40, 40)
)

pixelization = al.Pixelization(
    mesh=al.mesh.RectangularBilinearAdaptDensity(shape=mesh_shape),
    regularization=al.reg.Constant(coefficient=1.0),
)

source_galaxy = al.Galaxy(redshift=1.0, pixelization=pixelization)

fit = perform_fit_with_source_galaxy_mask_and_border(
    dataset=dataset_unmasked,
    source_galaxy=source_galaxy,
    mask=mask_annular,
    settings=al.Settings(use_border_relocator=False),
)

mapper = fit.inversion.cls_list_from(al.Mapper)[0]

aplt.plot_array(array=fit.model_data, title="Plane 1 Image")

Everything looks fine, we get a reconstructed source on a visually appeasing source-plane grid. So, why are we so
worried about borders? Lets see what happens if we use a circular mask instead.

In [ ]:
fit = perform_fit_with_source_galaxy_mask_and_border(
    dataset=dataset_unmasked,
    source_galaxy=source_galaxy,
    mask=mask_circular,
    settings=al.Settings(use_border_relocator=False),
)

mapper = fit.inversion.cls_list_from(al.Mapper)[0]

aplt.plot_array(
    array=fit.inversion.mapped_reconstructed_operated_data,
    title="Inversion Reconstruction",
)

Woah, whats happened? There are lots of additional $(y,x)$ coordinates in the source-plane grid, some of which trace 
to extremely large radii far away from the central regions of the source-plane! These points are the traced image-pixels 
that correspond to the central image-pixels that the annular mask removed (e.g. they were at radii with 0.8" of the 
centre).

Lets quickly check this by plotting the indexes of these image-pixels.

In [ ]:
mapper = fit.inversion.cls_list_from(al.Mapper)[0]


aplt.plot_grid(grid=mapper.source_plane_mesh_grid, title="Source-Plane Mesh Grid")

So, why is this happening? What is the mass profile physically doing to create these source plane coordinates at 
extremely large radial values? 

Towards the centre of the elliptical isothermal mass profile, the density begins to rise very sharply, it becomes
extremely steep or 'cuspy'. This cuspy behaviour towards its centre causes extremely large deflection angles to be
calculated.

Central image pixels are therefore subjected to 'demagnification', whereby they trace to extremely large values in
the source plane!

Physically, this is not a problem, and it is the reason we do not see a 'central image' in most strong lenses, as the 
light-rays which take this path through the centre of the lens are demagnified. However, if the lens galaxy had a less
steep inner mass distribution (termed a 'core') we would see the central image.

Demagnification is a problem for the pixelization and inversion though, which reconstruct the flux of these 
demagnified pixels just like the other pixels in the image-pixel. There are two negative consequences:

 1) The rectangular pixel-grid that we 'lay over' the source-plane becomes very large, because it expands to include
 the demagnified image-pixels. As a result, larger source-pixels are used to reconstruct the central regions of the
 source-plane (where the source galaxy is actually located), meaning we reconstruct the source-galaxy at a lower
 effective resolution.

 2) The inversion reconstructs the flux of the demagnified image pixels using source-pixels which contain *only*
 demagnified image pixels (these are the source pixels at the edge of the source plane). These source-pixels *should*
 have had other image-pixels traced within them via image-pixels at even larger radii from the centre of the lens
 galaxy. However, these image-pixels are at radii above 2.5", meaning the circular mask removed them from the inversion.

Lets quickly use a large circular mask to confirm that these pixels exist when we don't mask them.

In [ ]:
mask_circular_large = al.Mask2D.circular(
    shape_native=dataset_unmasked.shape_native,
    pixel_scales=dataset_unmasked.pixel_scales,
    radius=4.0,
)

fit = perform_fit_with_source_galaxy_mask_and_border(
    dataset=dataset_unmasked,
    source_galaxy=source_galaxy,
    mask=mask_circular_large,
    settings=al.Settings(use_border_relocator=False),
)

mapper = fit.inversion.cls_list_from(al.Mapper)[0]


aplt.plot_array(array=fit.model_data, title="Plane 1 Image")

This second point is a *huge* problem and it can introduce extremely dangerous systematics into our source
reconstruction and lens models.

Borders are the solution to this problem. We simply take the mask's` border in the image-plane that we showed above,
trace it to the source-plane and relocate all traced image-pixels pixels outside this source-plane border to its edge.
Lets take a look:

In [ ]:
fit = perform_fit_with_source_galaxy_mask_and_border(
    dataset=dataset_unmasked,
    source_galaxy=source_galaxy,
    mask=mask_circular,
    settings=al.Settings(use_border_relocator=True),
)

mapper = fit.inversion.cls_list_from(al.Mapper)[0]

aplt.plot_array(array=fit.model_data, title="Plane 1 Image")


aplt.plot_grid(grid=mapper.source_plane_mesh_grid, title="Source-Plane Mesh Grid")

This successfully addresses both of the issues above! However, you might be thinking, isn't that a bit of a hack? Its 
not really a physical treatment of the ray-tracing, is it?

Well, you are right. However, the *only* physical way to do this would be to use a mask so large that all demangified 
central pixels are surrounded by traced image-pixels. This would require a mask so large the **PyAutoLens** fit would
become extremely slow. This is not a feasible solution, thus borders provide us with a workaround, one that is 
extensively tested and does not introduce systematic biases into the lens modeling procedure.

To end, lets illustrate how important borders are when modeling multiple lens galaxies. Their complex mass distribution 
and lensing configurations produce nasty edge effects where image pixels not just in the centre of mask, but anywhere 
in the mask, trace beyond the source-plane border.

we'll use new strong lensing data, where:

 - The lens galaxy's light is omitted.
 - There are two lens galaxies whose `MassProfile`'s are `Isothermal`.
 - The source galaxy's light is an `Sersic`.

In [ ]:
dataset_name = "x2_lens_galaxies"
dataset_path = Path("dataset") / "imaging" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/simulator/lens_x2.py"],
        check=True,
    )

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    psf_path=dataset_path / "psf.fits",
    pixel_scales=0.1,
)

We again must define a mask around this image, lets start with a 2.8" mask. we'll vary the mask size to illustrate the
effects of the border in a moment.

In [ ]:
dataset_unmasked_x2 = dataset

mask_circular = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=2.8,
)

dataset = dataset.apply_mask(mask=mask_circular)

border = mask_circular.derive_grid.border


aplt.subplot_imaging_dataset(dataset=dataset)

We need to redefine our perform fit function, to use the x2 lens galaxy model.

In [ ]:


def perform_fit_x2_lenses_with_source_galaxy_mask_and_border(
    dataset, source_galaxy, mask, settings
):
    dataset = dataset.apply_mask(mask=mask)

    lens_galaxy_0 = al.Galaxy(
        redshift=0.5,
        bulge=al.lp.Sersic(
            centre=(0.0, -1.0),
            ell_comps=(0.25, 0.1),
            intensity=0.1,
            effective_radius=0.8,
            sersic_index=2.5,
        ),
        mass=al.mp.Isothermal(
            centre=(0.0, -1.0), ell_comps=(0.17647, 0.0), einstein_radius=1.0
        ),
    )

    lens_galaxy_1 = al.Galaxy(
        redshift=0.5,
        bulge=al.lp.Sersic(
            centre=(0.0, 1.0),
            ell_comps=(0.0, 0.1),
            intensity=0.1,
            effective_radius=0.6,
            sersic_index=3.0,
        ),
        mass=al.mp.Isothermal(
            centre=(0.0, 1.0), ell_comps=(0.0, -0.111111), einstein_radius=0.8
        ),
    )

    tracer = al.Tracer(galaxies=[lens_galaxy_0, lens_galaxy_1, source_galaxy])

    return al.FitImaging(dataset=dataset, tracer=tracer, settings=settings)


Now, lets fit this image using the input model and perform the source reconstruction without a border. As you can see, 
we get many demagnified image pixels which trace well beyond our source-plane border if we don't relocate them!

In [ ]:
fit = perform_fit_x2_lenses_with_source_galaxy_mask_and_border(
    dataset=dataset_unmasked_x2,
    source_galaxy=source_galaxy,
    mask=mask_circular,
    settings=al.Settings(use_border_relocator=False),
)

mapper = fit.inversion.cls_list_from(al.Mapper)[0]


aplt.plot_array(array=fit.model_data, title="Plane 1 Image")

However, when we relocate them, we get a good-looking source-plane with a well defined border and edge, thus ensuring
our analysis will be free of systematic biases.

In [ ]:
fit = perform_fit_x2_lenses_with_source_galaxy_mask_and_border(
    dataset=dataset_unmasked_x2,
    source_galaxy=source_galaxy,
    mask=mask_circular,
    settings=al.Settings(use_border_relocator=True),
)

mapper = fit.inversion.cls_list_from(al.Mapper)[0]

aplt.plot_array(array=fit.model_data, title="Plane 1 Image")

Multi-galaxy modeling is rife with border effects, and if you have multiple lens galaxies I heartily recommend you
keep a close eye on your source-plane borders!

Care must also be taken when choosing the size of your mask. If you don't choose a big enough mask, the border won't
be able to relocate all of the demagnified image pixels to the border edge.

The fits below repeat the analysis with progressively larger circular masks, so you can see how the mask size
changes the border and the resulting model image.

In [ ]:
dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    psf_path=dataset_path / "psf.fits",
    pixel_scales=0.1,
)
mask_circular = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=2.5,
)
fit = perform_fit_x2_lenses_with_source_galaxy_mask_and_border(
    dataset=dataset,
    source_galaxy=source_galaxy,
    mask=mask_circular,
    settings=al.Settings(use_border_relocator=True),
)

mapper = fit.inversion.cls_list_from(al.Mapper)[0]


aplt.plot_array(array=fit.model_data, title="Plane 1 Image")

mask_circular = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=2.7,
)
fit = perform_fit_x2_lenses_with_source_galaxy_mask_and_border(
    dataset=dataset,
    source_galaxy=source_galaxy,
    mask=mask_circular,
    settings=al.Settings(use_border_relocator=True),
)

mapper = fit.inversion.cls_list_from(al.Mapper)[0]


aplt.plot_array(array=fit.model_data, title="Plane 1 Image")


dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    psf_path=dataset_path / "psf.fits",
    pixel_scales=0.1,
)
mask_circular = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=2.9,
)
fit = perform_fit_x2_lenses_with_source_galaxy_mask_and_border(
    dataset=dataset,
    source_galaxy=source_galaxy,
    mask=mask_circular,
    settings=al.Settings(use_border_relocator=True),
)

mapper = fit.inversion.cls_list_from(al.Mapper)[0]


aplt.plot_array(array=fit.model_data, title="Plane 1 Image")


dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    psf_path=dataset_path / "psf.fits",
    pixel_scales=0.1,
)

mask_circular = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=3.1,
)
fit = perform_fit_x2_lenses_with_source_galaxy_mask_and_border(
    dataset=dataset,
    source_galaxy=source_galaxy,
    mask=mask_circular,
    settings=al.Settings(use_border_relocator=True),
)

mapper = fit.inversion.cls_list_from(al.Mapper)[0]


aplt.plot_array(array=fit.model_data, title="Plane 1 Image")

__Wrap Up__

To perform inversions, you probably won't think about borders all that often. Borders should  pretty much take care of 
themselves.

However, as I showed above, if you don't choose a large enough mask things can go wrong and its important you know what 
borders are, so you can diagnose this potential source of systematics!